# 3가지 Method별 RAG 검색 결과 비교

#### * 검색에 사용할 데이터를 3가지로 고려하여 임베딩 진행
1. 과거 사용자 질의만 검색에 활용했을 때
2. 과거 질의응답에 대한 요구사항만 검색에 활용했을 때
3. 과거 사용자 질의응답 정보를 모두 검색에 활용했을 때

In [ ]:
# --------------------------------------------------------
# Setting
# 라이브러리 로드 / DB 연결 / 모델 로드
# --------------------------------------------------------
import os
import psycopg2
from psycopg2.extras import RealDictCursor
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

POSTGRESQL_PORT = os.getenv("POSTGRESQL_PORT")
POSTGRESQL_USER = os.getenv("POSTGRESQL_USER")
POSTGRESQL_PASSWORD = os.getenv("POSTGRESQL_PASSWORD")
POSTGRESQL_DBNAME = os.getenv("POSTGRESQL_DBNAME")

DB_CONFIG = {
    "host": "localhost",
    "port": POSTGRESQL_PORT,
    "user": POSTGRESQL_USER,
    "password": POSTGRESQL_PASSWORD,
    "dbname": POSTGRESQL_DBNAME
}

conn = psycopg2.connect(**DB_CONFIG)
register_vector(conn)
cur = conn.cursor(cursor_factory=RealDictCursor)

model = SentenceTransformer("dragonkue/snowflake-arctic-embed-l-v2.0-ko")
print("DB 연결 및 모델 로드 완료")

In [ ]:
# --------------------------------------------------------
# RAG TEST - 코사인 유사도 기준 Top-3
# --------------------------------------------------------
def search_top3(user_query: str, embedding_column: str = "embedding_full"):
    # user_query: 사용자가 입력한 신규 질의 텍스트
    # embedding_column: embedding_q / embedding_i / embedding_full 중 선택

    
    # 1) 신규 질의 인코딩 (query 프롬프트 적용 - 저장된 데이터와 다르게 처리해야 함)
    query_vector = model.encode(user_query, prompt_name="query", normalize_embeddings=True)


    # 2) 코사인 거리 기준 Top-3 검색
    sql = f"""
        SELECT qa_id, full_source, consulting_category, consulting_topic, qa_topic, consulting_purpose,
               {embedding_column} <=> %s AS distance
        FROM financial_consulting_qa
        ORDER BY {embedding_column} <=> %s
        LIMIT 3;
    """

    cur.execute(sql, (query_vector, query_vector))
    return cur.fetchall()


def print_results(results):
    for i, row in enumerate(results, 1):
        print("---------------------------------------------------")
        print(f"[{i}순위] 거리: {row['distance']:.4f}")
        print(f"과거 질의응답:\n{row['full_source']}")
        print(f"metadata:\n- consulting_category={row['consulting_category']}\n- consulting_topic={row['consulting_topic']}\n- qa_topic={row['qa_topic']}\n- consulting_purpose={row['consulting_purpose']}")
        print("---------------------------------------------------")

# 질의set 
#### (활용 데이터셋 - validation 내 데이터 中 랜덤 선택)
1. 보험 - 중도 인출 신청 절차를 알려주세요. (21-1_ins_03_903666)
2. 보험 - 해외여행 중 핸드폰이 고장났는데 수리가 불가능한 상황이에요. 이런 경우 보험에서 보상이 어떻게 가능한지 구체적으로 알려주세요 (21-1_ins_05_902294)
3. 은행 - 청약 통장 해지 절차를 알려주세요 (21-1_bk_04_030321)
4. 은행 - 공항에서 현금 인출할 수 있나요? 또 인출이 가능하다면 인출 수수료는 얼마인가요? (21-1_bk_09_029600_001)
5. 증권 - 하루에 모바일 OTP 생성은 몇 번까지 가능한가요? (21-1_sec_01_012766)
6. 증권 - 성년 후견인 신청을 한 상태인데 계좌 해지를 하고 싶어요. 이때 필요한 서류를 알려주세요. (21-1_sec_07_014014)

In [ ]:
# 보험 - 중도 인출 신청 절차를 알려주세요. (21-1_ins_03_903666)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 중도 인출 신청 절차를 알려주세요.


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.1386
과거 질의응답:
요구사항:중도인출 신청 절차와 처리 가능 시점을 안내하시오.
고객질문:중도인출을 신청하려면 어떤 절차로 진행해야 하는지 알고 싶습니다.
상담사답변:중도인출을 신청하시려면 평일 영업시간 내에 연락주시면 즉시 처리 절차를 도와드립니다. 중도인출은 예약 방식이 따로 존재하지 않으며, 신청하시는 당일에 입금 처리까지 완료됩니다. 인출 가능한 금액은 조회된 지급금과 한도 기준으로 확인되며, 인출은 일정 금액 단위로 진행됩니다. 인출 신청 시 수수료가 적용되며, 수수료는 인출 금액에 비례하여 부과됩니다. 인출 후 잔여 금액에 대해서는 기존 이자 적용 방식이 그대로 유지됩니다. 원하시는 날짜가 있다면 해당 날짜에 영업시간 내 접수하시면 바로 진행 가능합니다.
꼬리질문:그럼 원하는 날짜에 맞춰 당일 중도인출 처리가 가능하다는 말씀이신가요?
종합답변:중도인출은 평일 영업시간 내에 신청하시면 별도 예약 없이 당일 입금이 가능합니다. 인출 금액과 수수료는 신청 시점의 조회 금액을 기준으로 처리되며, 원하시는 날짜에 연락 주시면 해당일에 중도인출을 도와드리겠습니다.
metadata:
- consulting_category=보험
- consulting_topic=계약내용변경/해지
- qa_topic=운전자/질병/상해보험등계약내용변경/해지
- consulting_purpose=중도인출 금액 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.1386
과거 질의응답:
요구사항:중도 인출 절차와 주의사항을 안내하시오.
고객질문:중도 인출을 조금 하려고 하는데, 구체적으로 어떤 절차를 거쳐야 하는지 자세히 알려주실 수 있나요?
상담사답변:전체 계좌 화면에서 왼쪽 메뉴를 선택하신 후 계좌 관리 항목으로 이동하시면 해지 예상 조회라는 메뉴가 나타납니다. 중도 인출을 진행할 경우 남은 계약 기간에 따라 이자 차감이 발생할 수 있는데, 이 차감 금액은 해지 예상 조회 화면에서 상세하게 확인하실 수 있도록 안내해 드리고 있습니다.
꼬리질문:그러면 그 이자 차감 금액이 생각보다 많이 발생하지는 않을까요? 어느 정도 수준인지 알려주실 수 있나요?
종합답변:먼저 스마트폰 앱에서 전체 계좌 화면에 진입하시고 좌측 메뉴를 선택해 계좌 관리 섹션으로 이동하십시오. 해당 화면에 표시되는 ‘해지 예상 조회’ 메뉴를 클릭하면 중도 인출 가능 여부와 남은 기간에 따른 이자 차감액을 확인할 수 있습니다. 차감액은 계약 시 약정된 이자율과 남은 기간을 기준으로 계산되며, 보통 전체 이자의 10% 내외가 차감됩니다. 화면에 표시된 금액을 검토한 후, 조건에 동의하시면 하단의 해지 신청 버튼을 눌러 진행하시기 바랍니다. 추가로 비대면 처리가 어려운 경우에는 가까운 지점을 방문해 신분증과 휴대폰을 지참하시면 신속히 처리됩니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도 인출 안내
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.1698
과거 질의응답:
요구사항:중도인출 절차와 조건을 안내하시오.
고객질문:중도인출을 하려면 어떤 절차를 거쳐야 하는지 그리고 그 조건들이 구체적으로 어떻게 되는지 자세하게 설명해 주실 수 있나요?
상담사답변:중도인출은 계약 해지 없이 이자 부담 없이 가능하지만, 인출 시점 이후 환급금이 줄어들 수 있다는 점을 유념해 주셔야 합니다. 인출 가능 금액은 해지환급금을 기준으로 산정되며, 계약 상태와 설정된 한도에 따라 달라지기 때문에 정확한 금액은 담당자가 확인 후 별도로 연락드려 절차와 준비 사항을 안내해 드리겠습니다.
꼬리질문:그러면 중도인출을 할 수 있는 구체적인 금액 한도와 함께 준비해야 할 서류나 절차에 대해 좀 더 상세하게 알려 주실 수 있을까요?
종합답변:중도인출은 계약 해지환급금 기준으로 계산되며, 이자를 부과하지 않고 인출할 수 있습니다. 다만 인출 금액만큼 향후 환급금이 감소하게 됩니다. 가능한 인출 금액은 현재 계약 상태와 설정된 한도에 따라 달라지며, 고객님의 경우 한도 내에서 요청하신 금액이 가능합니다. 중도인출을 진행하려면 먼저 본인 확인 절차를 거쳐야 하며, 확인이 완료되면 담당자가 전화로 연락드려 구체적인 신청 방법과 필요한 서류를 안내합니다. 신청 후에는 인출 금액이 환급금에서 차감되며, 이후 남은 금액에 대해 기존 계약 조건대로 보장이 유지됩니다.
metadata:
- consulting_category=보험
- consulting_topic=계약내용변경/해지
- qa_topic=운전자/질병/상해보험등계약내용변경/해지
- consulting_purpose=중도인출 절차 문의
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2134
과거 질의응답:
요구사항:중도 인출 절차와 조건을 안내하시오.
고객질문:급하게 돈이 필요한데, 중도에 인출하고 다시 재가입할 수 있나요.
상담사답변:일부 해지는 예금 가입도 두 번까지 가능하며, 두 차례까지 중간 인출이 허용됩니다. 최저 가입 금액은 일정 금액 이상이어야 하며, 그 이하에서는 부분 해지가 제한될 수 있습니다. 고객이 불입한 금액 중 일정 금액은 원금으로 남고, 나머지 금액을 두 번까지 중간에 인출할 수 있습니다. 인출 후에도 남은 금액은 기존 조건대로 유지됩니다. 또한 인출 시 적용되는 이자율은 계약 시 약정된 금리와 동일하며, 현재 수수료는 부과되지 않습니다. 모바일 앱을 통해 진행하면 편리합니다. 앱에 로그인 후 전체 계좌 메뉴에서 해당 예적금 상품을 선택하고, 계좌 관리에서 해지 예상 조회를 선택한 뒤 중도 인출 금액을 입력하고 확인을 누르면 절차가 완료됩니다.
꼬리질문:절차를 어떻게 진행하면 되나요.
종합답변:중도 인출이 가능한 경우, 예적금 상품을 두 번까지 부분 해지할 수 있으며, 최소 가입 금액을 유지해야 합니다. 인출 후 남은 금액은 기존 금리와 조건대로 유지되며, 이자율은 계약 시 약정된 금리와 동일합니다. 현재 수수료는 부과되지 않으니 안심하셔도 됩니다. 절차는 모바일 앱에서 진행하는 것이 가장 편리합니다. 먼저 앱에 로그인한 뒤 화면 상단의 전체 계좌 버튼을 눌러 계좌 목록을 확인합니다. 원하는 예적금 상품을 선택하고, 계좌 관리 메뉴로 이동합니다. 계좌 관리 화면에서 ‘해지 예상 조회’ 버튼을 클릭하고, 중도 인출 금액을 입력한 뒤 확인을 누르면 인출이 처리됩니다. 인출이 완료되면 남은 원금은 기존 조건대로 유지되며, 추가 불입이 필요할 경우 동일한 절차로 진행하시면 됩니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도 인출 안내
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.2134
과거 질의응답:
요구사항:중도 인출 절차와 조건을 안내하시오.
고객질문:목돈이 급히 필요해서 중도 인출을 하려고 하는데, 인출할 수 있는 금액의 한도와 함께 구체적인 절차에 대해 자세히 설명해 주실 수 있을까요?
상담사답변:만기일 이전에는 한 번만 일부 해지가 가능하며, 이때 적용되는 금리와 절차에 대해 상세히 안내해 드립니다. 인출하실 금액은 직접 선택할 수 있고, 최소 가입 금액은 ●●원 이상 남겨두셔야 하며, 관련 서류 작성과 확인 절차가 반드시 필요합니다.
꼬리질문:지금 시스템에서 중도 인출 옵션이 보이지 않는데, 이게 정상적인 상황인지 확인해 주실 수 있나요?
종합답변:중도 인출은 만기일 이전에 한 번만 가능하며, 고객님이 원하시는 금액만 선택하여 인출하실 수 있습니다. 인출 시 적용되는 금리는 중도 해지 금리 기준에 따라 차감되며, 인출 가능한 금액은 최소 가입 금액인 ●●원 이상을 계좌에 남겨야 합니다. 인출을 진행하려면 해당 서류를 작성하고 확인 절차를 거쳐야 하며, 온라인 등록이나 영업점 방문을 통해 신청하실 수 있습니다. 시스템에 옵션이 보이지 않을 경우, 이는 정상적인 절차이며 담당 부서에서 바로 확인해 드리겠습니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도 인출 절차
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.2134
과거 질의응답:
요구사항:중도 인출 절차와 조건을 안내하시오.
고객질문:현재 중도 인출에 대해 궁금한 점이 있습니다.
상담사답변:고객님은 한번만 출금을 일부 해지해서 사용하실 수 있습니다. ●●원 이상을 남겨놓으셔야 합니다. 해지할 금액을 입력하시면 됩니다. 해지 절차는 온라인으로도 가능하지만, 원금이 ●●원 이하이거나 시스템에 반영되지 않을 경우 영업점을 방문하셔야 할 수도 있습니다.
꼬리질문:그럼 어떻게 되는 건가요? 저는 저축금 한도 내에서 한번 인출할 수 있나요? 그렇게 해서 계좌 해지를 누르면 되는 건가요? 해지하는 금액만큼 이자가 계산되어 같이 나오나요?
종합답변:고객님께서는 중도 인출을 원하실 경우, 한 번만 출금을 일부 해지하여 사용하실 수 있습니다. 이때 계좌에 최소 ●●원 이상을 남겨두셔야 하며, 해지하고자 하는 금액을 입력하시면 됩니다. 해지 절차는 온라인으로 진행 가능하지만, 원금이 ●●원 이하이거나 시스템에 해지 요청이 반영되지 않을 경우 영업점을 직접 방문하셔야 할 수도 있습니다. 해지 금액에 대한 이자는 해당 금액에 대해 계산되어 함께 안내됩니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도 인출 안내
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3288
과거 질의응답:
요구사항:중도 인출 신청 방법과 금액, 이자 적용 기준을 안내하시오.
고객질문:중도 인출 시 필요한 서류는 무엇인가요? 남겨둔 금액은 유지하면서 일부만 인출할 수 있을까요?
상담사답변:중도 인출은 일부 해지로 처리되며, 신분증이 필요합니다. 영업점 방문 또는 모바일 앱을 통해 신청 가능하며 원하시는 금액만 지정하시면 됩니다. 모바일 앱을 통해 신청하실 경우 앱 메인 화면의 예적금 관리 메뉴에서 해당 ★★ 상품을 선택하신 뒤 중도 인출을 누르시면 됩니다. 메뉴가 보이지 않으면 최신 버전으로 업데이트해 주세요. 남겨두신 ●●원은 그대로 유지한 채 일부 금액만 인출할 수 있습니다.
꼬리질문:인출 가능한 최대 금액과 이자 계산 방식이 궁금합니다.
종합답변:중도 인출은 적금 일부 해지로 처리되며, 신청 시 신분증이 필요합니다. 영업점 방문 또는 모바일 앱의 예·적금 관리 메뉴에서 신청하실 수 있고, 인출 금액만큼만 출금되며 나머지 잔액은 기존 이자율로 계속 이자가 지급됩니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도인출 신청 방법
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3355
과거 질의응답:
요구사항:중도인출 신청 절차와 처리 가능 시점을 안내하시오.
고객질문:중도인출을 신청하려면 어떤 절차로 진행해야 하는지 알고 싶습니다.
상담사답변:중도인출을 신청하시려면 평일 영업시간 내에 연락주시면 즉시 처리 절차를 도와드립니다. 중도인출은 예약 방식이 따로 존재하지 않으며, 신청하시는 당일에 입금 처리까지 완료됩니다. 인출 가능한 금액은 조회된 지급금과 한도 기준으로 확인되며, 인출은 일정 금액 단위로 진행됩니다. 인출 신청 시 수수료가 적용되며, 수수료는 인출 금액에 비례하여 부과됩니다. 인출 후 잔여 금액에 대해서는 기존 이자 적용 방식이 그대로 유지됩니다. 원하시는 날짜가 있다면 해당 날짜에 영업시간 내 접수하시면 바로 진행 가능합니다.
꼬리질문:그럼 원하는 날짜에 맞춰 당일 중도인출 처리가 가능하다는 말씀이신가요?
종합답변:중도인출은 평일 영업시간 내에 신청하시면 별도 예약 없이 당일 입금이 가능합니다. 인출 금액과 수수료는 신청 시점의 조회 금액을 기준으로 처리되며, 원하시는 날짜에 연락 주시면 해당일에 중도인출을 도와드리겠습니다.
metadata:
- consulting_category=보험
- consulting_topic=계약내용변경/해지
- qa_topic=운전자/질병/상해보험등계약내용변경/해지
- consulting_purpose=중도인출 금액 문의
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3388
과거 질의응답:
요구사항:중도 인출 절차와 조건을 안내하시오.
고객질문:목돈이 급히 필요해서 중도 인출을 하려고 하는데, 인출할 수 있는 금액의 한도와 함께 구체적인 절차에 대해 자세히 설명해 주실 수 있을까요?
상담사답변:만기일 이전에는 한 번만 일부 해지가 가능하며, 이때 적용되는 금리와 절차에 대해 상세히 안내해 드립니다. 인출하실 금액은 직접 선택할 수 있고, 최소 가입 금액은 ●●원 이상 남겨두셔야 하며, 관련 서류 작성과 확인 절차가 반드시 필요합니다.
꼬리질문:지금 시스템에서 중도 인출 옵션이 보이지 않는데, 이게 정상적인 상황인지 확인해 주실 수 있나요?
종합답변:중도 인출은 만기일 이전에 한 번만 가능하며, 고객님이 원하시는 금액만 선택하여 인출하실 수 있습니다. 인출 시 적용되는 금리는 중도 해지 금리 기준에 따라 차감되며, 인출 가능한 금액은 최소 가입 금액인 ●●원 이상을 계좌에 남겨야 합니다. 인출을 진행하려면 해당 서류를 작성하고 확인 절차를 거쳐야 하며, 온라인 등록이나 영업점 방문을 통해 신청하실 수 있습니다. 시스템에 옵션이 보이지 않을 경우, 이는 정상적인 절차이며 담당 부서에서 바로 확인해 드리겠습니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=중도 인출 절차
---------------------------------------------------
'''

In [ ]:
# 보험 - 해외여행 중 핸드폰이 고장났는데 수리가 불가능한 상황이에요. 이런 경우 보험에서 보상이 어떻게 가능한지 구체적으로 알려주세요 (21-1_ins_05_902294)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 해외여행 중 핸드폰이 고장났는데 수리가 불가능한 상황이에요. 이런 경우 보험에서 보상이 어떻게 가능한지 구체적으로 알려주세요


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.1406
과거 질의응답:
요구사항:해외 여행 중 핸드폰 손상에 대한 보상 절차를 안내하시오.
고객질문:해외 여행보험으로 핸드폰이 고장 나서 수리를 못하고 있습니다. 보상 가능 여부를 알려 주세요.
상담사답변:먼저 수리 불가 확인서 발급이 필요합니다. 해당 확인서는 보험금 청구에 필수적인 서류이니 꼭 준비해 주시기 바랍니다. 특별 약관 가입 여부를 확인하고, 접수를 진행하겠습니다. 현재 고객님께 적용 가능한 특약이 있는지 검토하겠습니다. 보상 한도는 한 품목당 ●●●원이며, 자기부담금 ●●●원 공제 후 보상해 드립니다. 한도 초과 부분은 별도 안내드리겠습니다. 필요한 서류를 문자로 보내 주시면 됩니다. 문자 전송 시 파일 크기가 제한될 수 있으니 압축해서 보내 주시면 더욱 원활합니다. 과거에 청구한 적이 있는 경우 청구 이력이 현재 청구에 영향을 줄 수 있어 확인이 필요합니다.
꼬리질문:수리 불가능한 경우 어떻게 처리해야 하나요?
종합답변:먼저 수리가 불가능하다는 확인서가 필요하며, 보상 한도는 품목별로 정해져 있고 자기부담금을 제외한 금액을 지급해 드립니다. 과거 청구 이력이 있을 경우 현재 청구에 영향을 줄 수 있으니 확인해 주시고, 구매 연도와 모델 확인이 가능한 가입증명서와 함께 필요한 서류를 문자로 보내 주시면 보상 절차를 안내해 드리겠습니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=핸드폰 손상 보험금 청구 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.2306
과거 질의응답:
요구사항:해외여행보험 핸드폰 파손 청구 절차와 필요 서류를 안내하시오.
고객질문:해외여행을 하던 중에 제 핸드폰이 실수로 파손되어서 수리비를 보험으로 청구하고 싶은데, 이 경우 어떻게 진행하면 되는지 궁금합니다.
상담사답변:고객님께서 가입하신 보험의 해당 담보는 해외여행 중 발생한 핸드폰 파손에 대해 수리비를 청구할 수 있도록 보장하고 있습니다. 다만, 자기부담금으로 ●●●원이 공제된 후 초과하는 금액에 대해 보상이 이루어지며, 보상 한도는 ●●●원으로 제한되어 있습니다.
꼬리질문:그러면 수리비 청구를 위해 어떤 절차를 밟아야 하는지 구체적으로 알려 주시고, 제출해야 하는 서류에는 어떤 것들이 포함되는지도 안내해 주실 수 있나요?
종합답변:해외 여행 중 핸드폰 파손으로 인한 수리비는 해외여행보험 담보에 포함됩니다. 자기부담금 ●●●원을 공제한 후 초과 금액을 보상하며, 보상 한도는 ●●●원입니다. 청구 절차는 수리 업체에서 수리받은 후 영수증과 견적서를 준비하시고, 사고 발생일로부터 3년 이내에 해당 서류를 제출하시면 됩니다. 서류는 이메일 또는 문자로 안내된 주소로 보내주시면 됩니다. 청구 시 필요한 기본 서류는 수리 영수증, 견적서, 사고 사실을 확인할 수 있는 증빙 서류이며, 추가로 신분증 사본이 요구됩니다. 부분 수리 후 남은 부분을 한국에서 진행하셔도 되며, 이 경우에도 수리 영수증과 견적서를 함께 제출해 주시기 바랍니다. 청구가 확인되면 계좌 이체를 통해 7일 이내에 보상금이 입금되며, 입금 완료 시 문자로 안내드립니다. 청구 진행 상황은 마이페이지의 청구 관리 메뉴에서 실시간으로 확인하실 수 있습니다. 동일 보험 기간 내 동일 사고 유형에 대한 추가 청구는 제한되지만, 다른 유형의 손해가 발생하면 별도 청구가 가능합니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=핸드폰 파손 청구
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.2784
과거 질의응답:
요구사항:여행자보험에서 휴대폰 보상 한도를 안내하시오.
고객질문:해외여행을 계획하고 있는데, 혹시 여행 중에 휴대폰이 분실되거나 고장 났을 때 보상받을 수 있는 한도가 정확히 얼마인지 자세하게 알려주실 수 있을까요? 보상 범위와 한도에 대해 구체적인 설명을 듣고 싶습니다.
상담사답변:고객님께서 가입하신 해외 여행자 보험 약관을 살펴보면, 휴대폰 손해에 대해 보상받을 수 있는 최대 한도는 계약서에 명확히 명시된 ㅇㅇㅇ원이며, 이와 함께 자기부담금은 ㅇㅇㅇ원으로 정해져 있어 실제 보상금액 산정 시 참고하시면 도움이 될 것입니다.
꼬리질문:그렇다면 휴대폰 보상을 청구하려면 어떤 절차를 거쳐야 하는지 구체적으로 안내해 주실 수 있을까요? 필요한 서류나 신청 방법도 함께 알려 주시면 감사하겠습니다.
종합답변:해외 여행자 보험에서 휴대폰 손해에 대한 보상 한도는 계약서에 명시된 ㅇㅇㅇ원이며, 실제 지급액은 수리비에서 자기부담금 ㅇㅇㅇ원을 차감한 금액이 됩니다. 따라서 고객님이 실제로 받을 수 있는 금액은 수리비용에서 해당 자기부담금을 제외한 금액이며, 보상 한도 초과 시 추가 비용은 본인 부담이 됩니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=보상 한도 확인
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2694
과거 질의응답:
요구사항:해외 여행 중 핸드폰 손상에 대한 보상 절차를 안내하시오.
고객질문:해외 여행보험으로 핸드폰이 고장 나서 수리를 못하고 있습니다. 보상 가능 여부를 알려 주세요.
상담사답변:먼저 수리 불가 확인서 발급이 필요합니다. 해당 확인서는 보험금 청구에 필수적인 서류이니 꼭 준비해 주시기 바랍니다. 특별 약관 가입 여부를 확인하고, 접수를 진행하겠습니다. 현재 고객님께 적용 가능한 특약이 있는지 검토하겠습니다. 보상 한도는 한 품목당 ●●●원이며, 자기부담금 ●●●원 공제 후 보상해 드립니다. 한도 초과 부분은 별도 안내드리겠습니다. 필요한 서류를 문자로 보내 주시면 됩니다. 문자 전송 시 파일 크기가 제한될 수 있으니 압축해서 보내 주시면 더욱 원활합니다. 과거에 청구한 적이 있는 경우 청구 이력이 현재 청구에 영향을 줄 수 있어 확인이 필요합니다.
꼬리질문:수리 불가능한 경우 어떻게 처리해야 하나요?
종합답변:먼저 수리가 불가능하다는 확인서가 필요하며, 보상 한도는 품목별로 정해져 있고 자기부담금을 제외한 금액을 지급해 드립니다. 과거 청구 이력이 있을 경우 현재 청구에 영향을 줄 수 있으니 확인해 주시고, 구매 연도와 모델 확인이 가능한 가입증명서와 함께 필요한 서류를 문자로 보내 주시면 보상 절차를 안내해 드리겠습니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=핸드폰 손상 보험금 청구 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3643
과거 질의응답:
요구사항:여행자 보험의 핸드폰 파손 보상 가능여부를 확인하고 필요서류를 안내하시오.
고객질문:여행 중 핸드폰이 파손되었는데, 가입되어 있는 여행자 보험으로 보상이 가능한가요?
상담사답변:여행기간이 이천 이십이년 ●●월 ●●일에서 ●●월 ●●일 여행 기간이 확인되고 있습니다. 이 기간 중에 핸드폰이 파손되었고, 휴대폰 손해 보상 특별약관에 가입되어 있으면 접수가 가능하고 담보를 확인해서 안내드리겠습니다. 핸드폰 명의 확인 후 서류 제출 방법과 접수 기한에 대한 상세 안내를 문자로 발송해드리겠습니다. 확인 후 필요 서류를 준비해주세요.
꼬리질문:이후 진행상황은 어떻게 알 수 있나요?
종합답변:고객님께서 여행하신 기간 중 휴대폰이 파손되었으며, 휴대폰 손해 보상 특별약관에 가입되어 있을 경우 접수가 가능합니다. 명의 확인 후 서류 제출 방법과 접수 기한을 문자로 안내해드리니 확인하신 후 필요한 서류를 준비해 주시기 바랍니다. 접수된 서류는 심사를 거쳐 지급 여부가 결정되며, 심사 과정은 보통 1~2주 정도 소요됩니다. 진행 상황은 문자나 전화로 안내해드리며, 확인 서류는 접수 후 며칠 내에 우편으로 발송될 예정입니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=휴대폰 파손 보상 문의
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3722
과거 질의응답:
요구사항:여행자보험으로 휴대폰 파손 청구 가능 여부를 안내하시오.
고객질문:제가 가입한 여행자 보험에서 휴대폰이 파손되었을 때, 이와 관련된 손해 배상 청구가 가능한지 구체적으로 확인해 주실 수 있나요? 특히 어떤 조건이나 제한 사항이 있는지도 함께 알려 주시면 감사하겠습니다.
상담사답변:휴대폰 파손에 대한 보험 청구 가능 여부는 고객님께서 가입하신 여행자 보험의 약관과 보장 범위에 따라 달라질 수 있습니다. 우선 고객님의 보험 계약 내용을 상세히 확인한 후, 청구가 가능한지 여부와 함께 필요한 서류 및 절차에 대해서도 차근차근 안내해 드리겠습니다.
꼬리질문:그렇다면 해외 여행 중 휴대폰 파손에 대해 보상받을 수 있는 최대 한도 금액이 어느 정도인지도 구체적으로 알려 주실 수 있을까요? 추가로 보상 범위에 포함되는 항목도 궁금합니다.
종합답변:휴대폰 파손에 대한 청구는 고객님의 여행자 보험 계약에 포함된 보장 항목에 따라 가능합니다. 계약서를 확인한 결과, 파손 손해에 대해 일정 금액까지 보상이 이루어지며, 청구 절차는 사고 발생 후 사진과 영수증을 준비해 모바일 앱이나 웹사이트에서 청구서를 작성하면 됩니다. 자세한 보상 한도와 조건은 계약 내용에 명시되어 있으니, 필요 시 계약서 조항을 다시 한 번 확인하시기 바랍니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=청구 가능 여부
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2404
과거 질의응답:
요구사항:해외 여행 중 핸드폰 손상에 대한 보상 절차를 안내하시오.
고객질문:해외 여행보험으로 핸드폰이 고장 나서 수리를 못하고 있습니다. 보상 가능 여부를 알려 주세요.
상담사답변:먼저 수리 불가 확인서 발급이 필요합니다. 해당 확인서는 보험금 청구에 필수적인 서류이니 꼭 준비해 주시기 바랍니다. 특별 약관 가입 여부를 확인하고, 접수를 진행하겠습니다. 현재 고객님께 적용 가능한 특약이 있는지 검토하겠습니다. 보상 한도는 한 품목당 ●●●원이며, 자기부담금 ●●●원 공제 후 보상해 드립니다. 한도 초과 부분은 별도 안내드리겠습니다. 필요한 서류를 문자로 보내 주시면 됩니다. 문자 전송 시 파일 크기가 제한될 수 있으니 압축해서 보내 주시면 더욱 원활합니다. 과거에 청구한 적이 있는 경우 청구 이력이 현재 청구에 영향을 줄 수 있어 확인이 필요합니다.
꼬리질문:수리 불가능한 경우 어떻게 처리해야 하나요?
종합답변:먼저 수리가 불가능하다는 확인서가 필요하며, 보상 한도는 품목별로 정해져 있고 자기부담금을 제외한 금액을 지급해 드립니다. 과거 청구 이력이 있을 경우 현재 청구에 영향을 줄 수 있으니 확인해 주시고, 구매 연도와 모델 확인이 가능한 가입증명서와 함께 필요한 서류를 문자로 보내 주시면 보상 절차를 안내해 드리겠습니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=핸드폰 손상 보험금 청구 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3153
과거 질의응답:
요구사항:해외여행보험 핸드폰 파손 청구 절차와 필요 서류를 안내하시오.
고객질문:해외여행을 하던 중에 제 핸드폰이 실수로 파손되어서 수리비를 보험으로 청구하고 싶은데, 이 경우 어떻게 진행하면 되는지 궁금합니다.
상담사답변:고객님께서 가입하신 보험의 해당 담보는 해외여행 중 발생한 핸드폰 파손에 대해 수리비를 청구할 수 있도록 보장하고 있습니다. 다만, 자기부담금으로 ●●●원이 공제된 후 초과하는 금액에 대해 보상이 이루어지며, 보상 한도는 ●●●원으로 제한되어 있습니다.
꼬리질문:그러면 수리비 청구를 위해 어떤 절차를 밟아야 하는지 구체적으로 알려 주시고, 제출해야 하는 서류에는 어떤 것들이 포함되는지도 안내해 주실 수 있나요?
종합답변:해외 여행 중 핸드폰 파손으로 인한 수리비는 해외여행보험 담보에 포함됩니다. 자기부담금 ●●●원을 공제한 후 초과 금액을 보상하며, 보상 한도는 ●●●원입니다. 청구 절차는 수리 업체에서 수리받은 후 영수증과 견적서를 준비하시고, 사고 발생일로부터 3년 이내에 해당 서류를 제출하시면 됩니다. 서류는 이메일 또는 문자로 안내된 주소로 보내주시면 됩니다. 청구 시 필요한 기본 서류는 수리 영수증, 견적서, 사고 사실을 확인할 수 있는 증빙 서류이며, 추가로 신분증 사본이 요구됩니다. 부분 수리 후 남은 부분을 한국에서 진행하셔도 되며, 이 경우에도 수리 영수증과 견적서를 함께 제출해 주시기 바랍니다. 청구가 확인되면 계좌 이체를 통해 7일 이내에 보상금이 입금되며, 입금 완료 시 문자로 안내드립니다. 청구 진행 상황은 마이페이지의 청구 관리 메뉴에서 실시간으로 확인하실 수 있습니다. 동일 보험 기간 내 동일 사고 유형에 대한 추가 청구는 제한되지만, 다른 유형의 손해가 발생하면 별도 청구가 가능합니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=핸드폰 파손 청구
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3316
과거 질의응답:
요구사항:여행자보험에서 휴대폰 보상 한도를 안내하시오.
고객질문:해외여행을 계획하고 있는데, 혹시 여행 중에 휴대폰이 분실되거나 고장 났을 때 보상받을 수 있는 한도가 정확히 얼마인지 자세하게 알려주실 수 있을까요? 보상 범위와 한도에 대해 구체적인 설명을 듣고 싶습니다.
상담사답변:고객님께서 가입하신 해외 여행자 보험 약관을 살펴보면, 휴대폰 손해에 대해 보상받을 수 있는 최대 한도는 계약서에 명확히 명시된 ㅇㅇㅇ원이며, 이와 함께 자기부담금은 ㅇㅇㅇ원으로 정해져 있어 실제 보상금액 산정 시 참고하시면 도움이 될 것입니다.
꼬리질문:그렇다면 휴대폰 보상을 청구하려면 어떤 절차를 거쳐야 하는지 구체적으로 안내해 주실 수 있을까요? 필요한 서류나 신청 방법도 함께 알려 주시면 감사하겠습니다.
종합답변:해외 여행자 보험에서 휴대폰 손해에 대한 보상 한도는 계약서에 명시된 ㅇㅇㅇ원이며, 실제 지급액은 수리비에서 자기부담금 ㅇㅇㅇ원을 차감한 금액이 됩니다. 따라서 고객님이 실제로 받을 수 있는 금액은 수리비용에서 해당 자기부담금을 제외한 금액이며, 보상 한도 초과 시 추가 비용은 본인 부담이 됩니다.
metadata:
- consulting_category=보험
- consulting_topic=보험금청구
- qa_topic=운전자/질병/상해보험등보험금청구/확인
- consulting_purpose=보상 한도 확인
---------------------------------------------------
'''

In [ ]:
# 은행 - 청약 통장 해지 절차를 알려주세요 (21-1_bk_04_030321)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 청약 통장 해지 절차를 알려주세요


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.1172
과거 질의응답:
요구사항:청약통장 해지 절차와 재가입 제한에 대해 상세히 안내하시오.
고객질문:청약 통장을 해지하려고 하는데, 구체적으로 어떤 절차를 따라야 하는지 자세히 알려주실 수 있나요?
상담사답변:청약 통장이 주택청약 종합 저축 계좌인지 먼저 확인해 주시기 바랍니다. 해지를 원하시면 인터넷뱅킹이나 스마트폰 뱅킹을 통해 간편하게 진행할 수 있으며, 인터넷뱅킹에서는 상단 메뉴 중 조회 항목에서 해지할 계좌를 선택한 뒤 해지 신청 버튼을 눌러 주시면 됩니다. 이후 공인인증서와 보안 매체 인증 절차를 완료하면 해지 처리가 정상적으로 이루어집니다.
꼬리질문:인터넷이나 스마트폰 뱅킹에서 해지할 때 구체적인 절차나 주의해야 할 점이 더 있을까요?
종합답변:청약통장을 해지하려면 먼저 고객님의 계좌가 주택청약 종합 저축 계좌인지 확인해 주시기 바랍니다. 확인이 끝나면 인터넷뱅킹에 로그인한 뒤 화면 상단의 조회 메뉴를 클릭합니다. 조회 메뉴에서 ‘해지 계좌 조회’ 또는 ‘해지 예상 조회’를 선택하고, 해지하고자 하는 청약통장을 선택합니다. 이후 화면에 표시된 ‘해지 신청’ 버튼을 눌러 진행합니다. 공인인증서와 보안 매체 인증을 정상적으로 완료하면 해지가 완료되며, 해지 완료 후 확인 메일이 발송되고 계좌는 즉시 정지됩니다. 재가입을 원하실 경우 동일한 명의로 6개월 이후에 가능하니 참고하시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약통장 해지
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.1234
과거 질의응답:
요구사항:청약 통장 해지 절차와 어플 이용 방법을 안내하시오.
고객질문:청약 통장을 해지하고 싶은데요. 구체적으로 어떤 절차를 따라야 하는지 자세히 알려주실 수 있을까요?
상담사답변:전화로는 해당 상품을 바로 해지해드릴 수 없어서 고객님께서 ★★은행 어플에 로그인하실 수 있는지 확인하고 싶습니다. 스마트폰에 어플을 아직 설치하지 않으셨다면 앱스토어나 구글플레이에서 다운로드 후 설치해 주시기 바랍니다. 어플에 로그인하신 뒤 화면 상단 메뉴에서 해지 옵션을 선택하시면, 단계별로 해지 경로를 자세히 안내해 드리겠습니다.
꼬리질문:사실 제가 스마트폰에 익숙하지 않아서 로그인 방법을 잘 모르겠어요. 방금 말씀드린 대로 어플에 로그인해서 해지 페이지까지 들어갔습니다.
종합답변:전화로는 상품 해지가 어려워 먼저 은행 앱에 로그인하실 수 있는지 확인 부탁드립니다. 앱이 없으시다면 앱스토어나 구글플레이에서 설치하신 후 로그인하셔서 상단 메뉴의 해지 옵션을 선택하시면 단계별로 안내해 드리겠습니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 해지 안내
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.1273
과거 질의응답:
요구사항:청약통장 해지 절차와 필요 서류, 처리 기간을 안내하시오.
고객질문:청약 통장을 해지하려고 하는데, 어떤 절차를 거쳐야 하는지 자세히 알려주실 수 있나요?
상담사답변:고객님께서 보유하신 통장이 주택청약종합저축인지 아니면 청년주택드림청약통장인지 먼저 확인이 필요합니다. 청년주택드림청약통장은 온라인이나 전화로 해지가 불가능하며, 반드시 신분증과 관련 서류를 지참하시고 가까운 영업점을 방문하셔야 합니다. 방문 후 서류 확인과 해지 절차가 진행되며, 처리 완료까지 영업일 기준으로 약 3~5일 정도 소요됩니다. 해지 시 별도의 수수료는 발생하지 않으나, 이후 재가입 시에는 신규 가입 조건이 적용된다는 점 참고 부탁드립니다.
꼬리질문:그러면 영업점에 가져가야 하는 서류는 구체적으로 어떤 것들이 필요한지 알려주실 수 있나요?
종합답변:청약 통장을 해지하시려면 먼저 고객님의 통장이 청년 주택 드림 청약 통장인지 확인해 주시기 바랍니다. 해당 통장은 온라인이나 전화로 해지가 불가능하므로, 신분증과 본인 통장을 지참하신 후 영업점에 방문하셔야 합니다. 방문 시 제출하실 서류는 신분증과 본인 통장이며, 통장에 도장이 필요할 경우 차등 적용됩니다. 영업점에서 서류를 확인하고 처리하는 데 영업일 기준 3~5일 정도 소요되며, 처리 완료 시 문자로 안내드릴 예정입니다. 해지 시 별도의 수수료는 부과되지 않으니 안심하시기 바랍니다. 해지 후 재가입을 원하실 경우 기존 우대 금리 혜택은 적용되지 않으며, 신규 조건에 따라 금리와 우대가 달라질 수 있습니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약통장 해지
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2028
과거 질의응답:
요구사항:청약 통장 해지 절차와 남은 금액 처리 방법을 안내하시오.
고객질문:저희 어머니가 청약 통장이 있는데, 병원에 계셔서 은행을 방문하기 어려워요. 해지를 하려면 어떻게 해야 하나요? 방법을 문자로 보내주실 수 있을까요?
상담사답변:요즘에는 방문 없이도 가능합니다. 스마트폰 저희 ★★은행 어플이나 인터넷 PC를 통해 직접 해지하실 수 있는 경로도 있습니다. 일반적으로 본인 명의 휴대폰이 있으시고, ★★은행에 거래하시는 입출금 계좌가 있는지 확인합니다. 연결이 되어 있지 않다면 회원 가입을 통해 직접 해지하거나 입출금 계좌로 입금받는 방법도 있습니다. 문자에 포함될 내용은 해지 신청 절차, 필요한 서류 안내, 진행 상황 확인 방법이며, 추가 문의 사항이 있으면 언제든 전화 주시라고 안내드리겠습니다.
꼬리질문:해지 후 청약 통장에 남은 금액은 어떻게 처리되나요?
종합답변:요즘에는 방문 없이 스마트폰 앱이나 인터넷 PC를 통해 직접 해지하실 수 있으며, 본인 명의 휴대폰과 거래 중인 입출금 계좌가 있는지 확인해 주시기 바랍니다. 해지 절차와 필요한 서류, 진행 상황 안내는 문자로 안내드리며, 남은 금액은 지정하신 계좌로 자동 이체되고 완료 후 확인 문자도 발송해드리니 궁금하신 점은 언제든 문의해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 통장 해지
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.2041
과거 질의응답:
요구사항:청약 통장 해지 절차와 주의사항을 안내하시오.
고객질문:청약 통장을 앱에서 해지할 수 있습니까? 절차를 알려 주세요.
상담사답변:청약을 만드셨을 때 영업점 영문을 통해서 진행하셨는지 비대면으로 신청하셨는지 확인 시 비대면으로 개설하셨으며, 잔액이 ●● ●원 이하이셔서 어플을 통해서도 진행 가능합니다. 따라서 앱에서 해지 요청을 진행하실 수 있으며, 절차는 메뉴, 통장 관리, 해지 신청 순서대로 진행됩니다. 메뉴에서 통장 관리 항목을 선택하고 해지 신청을 클릭하면 해지 요청이 접수됩니다. 해지 처리 후 남은 잔액이 있으면 고객님이 사전에 지정한 계좌로 자동 이체됩니다. 
꼬리질문:해지 완료 후에도 통장 관련 알림이 오나요?
종합답변:고객님께서 비대면으로 청약을 개설하셨고 잔액이 일정 금액 이하이므로 앱에서 메뉴의 통장 관리에서 해지 신청을 하시면 해지 요청이 접수됩니다. 해지 후 남은 잔액은 지정하신 계좌로 자동 이체되며, 통장 관련 알림은 해지 완료 시 중단되니 필요하시면 알림 설정을 변경해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 통장 해지
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.2064
과거 질의응답:
요구사항:청약 통장 해지 절차와 필요 서류를 안내하시오.
고객질문:현재 청약 통장에 매월 일정 금액이 자동으로 입금되고 있는 것으로 확인됩니다. 해당 입금 내역을 확인하고, 청약 통장을 해지하고 싶습니다. 해지를 위해서는 어떤 절차가 필요하며, 신분증만 지참하면 가능한지 알려주시기 바랍니다.
상담사답변:고객님, 먼저 본인 확인을 위해 계좌번호와 성함을 확인하였습니다. 청약 통장은 고객님 명의이므로 해지가 가능합니다. 해지를 원하실 경우, 신분증을 지참하시고 영업시간인 오후 4시까지 영업점에 방문해 주시기 바랍니다. 방문 시 담당 직원에게 청약 통장 해지를 요청하시면, 서류 확인 후 해지 절차가 진행됩니다. 해지 후 잔액은 고객님 계좌로 이체되며, 이체 완료까지는 영업일 기준으로 다소 시간이 소요될 수 있습니다.
꼬리질문:해지 절차가 완료된 후에 남은 금액을 바로 찾을 수 있는지, 그리고 추가로 준비해야 할 서류가 있는지 알려주시겠습니까?
종합답변:고객님께서는 청약 통장에 자동 입금되는 금액을 확인하셨으며, 이를 해지하고자 하십니다. 상담사는 먼저 고객님의 계좌번호와 성함을 확인하고, 청약 통장은 명의자 본인만 해지가 가능함을 안내했습니다. 해지를 위해서는 신분증을 지참하고 영업시간인 오후 4시까지 영업점에 방문해야 하며, 방문 시 담당 직원에게 해지를 요청하면 서류 확인 후 절차가 진행된다고 설명했습니다. 해지 후 남은 금액은 고객님의 계좌로 이체되며, 이체 완료까지 영업일 기준으로 시간이 소요될 수 있음을 알려드렸습니다.
metadata:
- consulting_category=은행
- consulting_topic=거래내역/잔액조회
- qa_topic=거래내역/잔액조회
- consulting_purpose=청약 통장 해지
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2280
과거 질의응답:
요구사항:청약통장 해지 절차와 유의사항을 상세히 안내하시오.
고객질문:제가 청약을 든 게 있는데, 혹시 해지를 할 수 있을까요?
상담사답변:현재 고객님의 청약통장은 해지 신청 전 사전 확인 절차가 필요합니다. 신청 가능 여부와 필요한 서류에 대해 자세히 안내드리겠습니다. 아쉽게도 유선상으로는 해지가 어렵습니다. 고객님 비밀번호가 등록되어 있고, 일정 금액 이하가 입금되어 있는 경우 모바일 어플을 통해 직접 해지를 하시거나, 가까운 지점에 신분증을 지참하고 방문하여 해지 요청을 해주셔야 합니다. 모바일 어플에서 해지 진행 시, 화면 하단의 청약관리 메뉴를 선택하고 계좌해지 옵션을 눌러 진행하시면 됩니다.
꼬리질문:해지 처리에는 보통 영업일 기준 1~2일 정도 소요된다고 들었는데, 정확한 소요 시간을 알려주실 수 있나요?
종합답변:청약통장을 해지하려면 먼저 사전 확인 절차를 거쳐야 합니다. 고객님의 비밀번호와 입금 금액이 기준에 부합하는지 확인한 후, 모바일 어플을 이용해 청약관리 메뉴에서 계좌해지 옵션을 선택하시거나, 신분증을 지참하고 가까운 지점에 방문하여 해지 요청을 하셔야 합니다. 해지 신청이 완료되면 영업일 기준 1~2일 이내에 처리됩니다. 처리 기간 중에는 계좌에 새로운 입출금이 발생하지 않도록 주의해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 해지 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.2310
과거 질의응답:
요구사항:청약 통장 해지 절차와 해지 후 이메일 발송 여부를 안내하시오.
고객질문:제가 ★★은행에 청약 주택 청약이 있는데, 그 청약을 취소하려고 합니다. 해지 절차는 어떻게 진행해야 하나요?
상담사답변:청약 통장을 해지하시려면 스마트폰 앱 또는 인터넷뱅킹에서 ‘전체 계좌’ 메뉴로 들어가신 뒤, 주택청약종합저축 계좌를 선택해 ‘계좌 관리’ 항목을 눌러 주시면 됩니다. 화면에서 ‘해지 예상 조회’로 이동하면 해지 가능 금액과 원금·이자를 확인할 수 있으며, 하단의 해지 버튼을 누르면 절차가 진행됩니다. 이후 안내에 따라 해지 사유를 선택하고, 계좌 비밀번호 입력과 스마트폰 간편 인증을 완료하시면 해지가 확정됩니다. 현재 위약금은 부과되지 않으며 잔액 전액이 반환 처리됩니다. 또한 해지 완료 후에는 등록된 이메일 주소로 해지 확인 메일이 자동 발송됩니다.
꼬리질문:해지 후 발송되는 확인서를 이메일로 받을 수 있나요?
종합답변:청약 통장은 앱이나 인터넷뱅킹에서 계좌 관리 메뉴를 통해 해지 예상 조회 후 해지하기를 선택하시면 비밀번호 입력과 간편 인증 절차를 거쳐 해지가 완료되며, 위약금은 부과되지 않습니다. 해지 완료 시 등록하신 이메일로 안내 메일이 자동 발송되니 이메일 주소가 변경되셨다면 미리 수정해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 해지
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.2435
과거 질의응답:
요구사항:청약 통장 해지 절차와 금액 입금 시점 및 세금 처리에 대해 안내하시오.
고객질문:그럼 어떻게 해야 되나요?
상담사답변:먼저 ★★은행 어플에 로그인하시고, 화면 상단의 전체 계좌를 선택하십시오. 전체 계좌 화면에서 고객님의 청약 통장을 찾아 계좌 관리를 클릭합니다. 계좌 관리 메뉴에서 해지 예상 조회를 선택하면 해지 진행 화면이 나타납니다. 해지 예상 조회 화면에서 금액과 해지 일자를 확인한 후 해지 버튼을 누르십시오. 해지 사유 입력란에 기타를 선택하고 간단히 이유를 적으시면 됩니다. 이후 안내에 따라 다음 및 계속 버튼을 차례대로 눌러 절차를 완료합니다. 해지 처리는 영업일 기준 1~2일 정도 소요되며, 완료 후 문자로 안내드릴 예정입니다.
꼬리질문:언제쯤 금액이 입금되는지 궁금합니다. 세금은 어떻게 되는지 알려 주세요.
종합답변:청약 통장 해지를 원하시는 경우, 먼저 ★★은행 어플에 로그인하시고 화면 상단의 전체 계좌를 선택하십시오. 전체 계좌 화면에서 청약 통장을 찾아 계좌 관리를 클릭한 뒤, 해지 예상 조회를 선택합니다. 해지 예상 조회 화면에서 금액과 해지 일자를 확인하고 해지 버튼을 누릅니다. 해지 사유 입력란에 기타를 선택하고 간단히 이유를 적은 후, 다음 및 계속 버튼을 차례대로 눌러 절차를 마무리합니다. 해지 처리는 영업일 기준 1~2일 정도 소요되며, 완료 후 문자로 안내드릴 예정입니다. 해지 후 원금과 이자는 지정하신 계좌로 자동 이체되며, 이자 계산은 해지일 기준 일할 계산이 적용되고 원천징수세 등 세금이 별도로 공제될 수 있습니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=청약 통장 해지
---------------------------------------------------
'''

In [ ]:
# 은행 - 공항에서 현금 인출할 수 있나요? 또 인출이 가능하다면 인출 수수료는 얼마인가요? (21-1_bk_09_029600_001)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 공항에서 현금 인출할 수 있나요? 또 인출이 가능하다면 인출 수수료는 얼마인가요?


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3155
과거 질의응답:
요구사항:환전 수수료와 현금 인출 가능 여부에 대해서 안내하시오.
고객질문: 환전 금액에 따라 환전 수수료가 달라 진다면 수수료는 대략 어느 정도인지, 그리고 환전 후에 바로 현금 인출이 가능한가요?
상담사답변:달러 기준으로는 보통 0.5% 정도이며, 소액일 경우 최소 수수료가 적용될 수 있습니다. 정확한 금액은 환전 금액을 입력하면 자동으로 계산됩니다. 수령하신 현금을 바로 인출하실 수 있습니다. 다만, 인출 금액이 큰 경우 신분증 확인 후, 거래 내역을 기록하고 필요 시 담당 직원과 대면 확인을 진행하며, 이 과정은 보통 5분 내외로 끝납니다.
꼬리질문:환전이 완료된 후에 확인 문자나 이메일을 받게 되나요?
종합답변:달러 환전 시 보통 0.5% 정도의 수수료가 부과되며, 소액일 경우 최소 수수료가 적용될 수 있습니다. 환전 금액을 입력하시면 정확한 수수료가 자동으로 계산되며, 현금 수령 후 바로 인출하실 수 있으나 큰 금액은 신분증 확인과 거래 내역 기록, 필요 시 담당 직원과 대면 확인 절차가 있으며 보통 5분 이내에 완료됩니다. 수령이 완료되면 등록된 휴대폰 번호와 이메일로 환전 금액, 수령 일시, 지점 정보가 포함된 확인 메시지를 보내드립니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=환전 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3236
과거 질의응답:
요구사항:고객이 해외 현금 인출 가능 여부와 수수료·환율, 인출 한도 설정에 대해 문의한 내용을 토대로, 해외 ATM 이용 방법과 수수료 확인 절차, 한도 조정 방법을 구체적으로 안내하시오.
고객질문:해외에서 현금 인출이 가능한가요? 인출 시 수수료와 적용 환율이 궁금합니다. 또한 인출 한도를 500만 원으로 설정하고 싶습니다.
상담사답변:해외 현금 인출은 현지 ATM에서 카드로 이용할 수 있습니다. 각 현지 은행별로 부과되는 수수료와 적용 환율이 다르며, 이용 전 모바일 앱의 ‘해외 인출 수수료 안내’ 메뉴에서 확인하실 수 있습니다. 인출 한도는 별도로 조정 가능하며, 요청하신 대로 500만 원으로 설정 접수되었습니다. 조정이 완료되면 문자로 결과가 발송되며, 이후 한도 내에서 자유롭게 인출이 가능합니다. 필요 시 한도 변경은 다시 요청하실 수 있습니다.
꼬리질문:한도 조정이 완료되었는지 확인하고 싶습니다.
종합답변:해외 현금 인출은 현지 ATM에서 카드로 가능하며, 국가와 은행에 따라 수수료와 환율이 다를 수 있으니 모바일 뱅킹 앱의 해외 인출 예상 수수료 메뉴에서 미리 확인해 주시기 바랍니다. 인출 한도는 요청하신 대로 500만 원으로 조정해드렸으며, 변경 완료 시 등록된 휴대폰 번호로 안내 문자가 발송되니 문자나 앱의 인출 한도 메뉴에서 확인해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=해외 인출 및 한도
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3284
과거 질의응답:
요구사항:현금 출금 절차와 주의사항을 안내하시오.
고객질문:현금 인출이 가능한지 여부와 함께, 구체적인 절차가 어떻게 되는지 자세히 알려주실 수 있나요?
상담사답변:가까운 은행 창구를 방문하시면 직접 현금 출금이 가능하며, 평일 오전 9시부터 오후 6시까지 영업하고 있습니다. 출금 시에는 반드시 신분증을 지참하셔야 하며, 출금액에 따라 수수료가 부과될 수 있고, 하루 출금 한도도 적용된다는 점 참고해 주시기 바랍니다. 만약 통장이 없으시다면 비대면 인증 절차를 거치거나 가까운 지점을 방문하여 추가 안내를 받으셔야 합니다.
꼬리질문:통장이 없는 상태에서는 계좌 조회와 현금 출금 절차를 구체적으로 어떻게 진행해야 하는지 다시 한번 설명해 주실 수 있나요?
종합답변:현금 출금을 원하시면 가까운 지점 창구를 방문하시고, 평일 09시부터 18시까지 영업하는 시간에 신분증을 지참해 주십시오. 출금 시에는 수수료가 부과되며, 일일 출금 한도가 적용됩니다. 통장이 없으신 경우, 비대면 인증 절차를 통해 계좌 조회와 출금을 진행하거나 직접 지점을 방문하여 담당 직원의 안내를 받으시면 됩니다.
metadata:
- consulting_category=증권
- consulting_topic=자금이체/계좌제한
- qa_topic=자금이체/계좌제한
- consulting_purpose=현금 출금 안내
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3138
과거 질의응답:
요구사항:환전 수수료와 현금 인출 가능 여부에 대해서 안내하시오.
고객질문: 환전 금액에 따라 환전 수수료가 달라 진다면 수수료는 대략 어느 정도인지, 그리고 환전 후에 바로 현금 인출이 가능한가요?
상담사답변:달러 기준으로는 보통 0.5% 정도이며, 소액일 경우 최소 수수료가 적용될 수 있습니다. 정확한 금액은 환전 금액을 입력하면 자동으로 계산됩니다. 수령하신 현금을 바로 인출하실 수 있습니다. 다만, 인출 금액이 큰 경우 신분증 확인 후, 거래 내역을 기록하고 필요 시 담당 직원과 대면 확인을 진행하며, 이 과정은 보통 5분 내외로 끝납니다.
꼬리질문:환전이 완료된 후에 확인 문자나 이메일을 받게 되나요?
종합답변:달러 환전 시 보통 0.5% 정도의 수수료가 부과되며, 소액일 경우 최소 수수료가 적용될 수 있습니다. 환전 금액을 입력하시면 정확한 수수료가 자동으로 계산되며, 현금 수령 후 바로 인출하실 수 있으나 큰 금액은 신분증 확인과 거래 내역 기록, 필요 시 담당 직원과 대면 확인 절차가 있으며 보통 5분 이내에 완료됩니다. 수령이 완료되면 등록된 휴대폰 번호와 이메일로 환전 금액, 수령 일시, 지점 정보가 포함된 확인 메시지를 보내드립니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=환전 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3431
과거 질의응답:
요구사항:해외 환전 현금 수령 가능 여부와 적용되는 수수료, 공항환전 절차를 안내하시오.
고객질문:환전을 현금으로 받고 싶습니다. 직접 지점에 가서 체크카드만 있으면 받을 수 있는지 ,수수료,  국내에서 현금을 받는 방법에 대해서 궁금합니다.
상담사답변:현금 수령을 원하실 경우, 일반적인 환전 서비스는 해외 현지에서만 가능하고, 국내에서는 전자 형태로만 제공됩니다.수수료는 환전 금액의 0.5%가 부과되며, 추가로 현금 인출 시 해외 ATM 이용 시 발생하는 현지 수수료가 별도로 청구될 수 있음을 알려드립니다.외화 머니로, 백프로 환율 우대를 받으면서 충전한 경우 국내에서 출금이나 수령이 불가능합니다. 발급받은 ★★ 카드를 해외 현지에서 사용해 등기소에서 인출하거나 결제 시 충전됩니다. 
꼬리질문:공항에서 환전 후 현금으로 받은 적이 있는데, 그 절차가 어떻게 되나요?
종합답변:고객님께서는 현금 수령 가능 여부와 체크카드 이용에 대해 문의하셨습니다. 국내에서는 외화 환전이 전자 방식으로만 가능하며 현금 수령은 해외 현지에서만 가능하니 참고해 주시기 바랍니다. 환전 시 0.5% 수수료가 부과되며 해외 ATM 이용 시 추가 현지 수수료가 발생할 수 있습니다. 공항에서 환전하실 경우 앱에서 통화를 선택하고 미리 예약하시면 지정한 수령점에서 신분증을 제시하여 현금을 받으실 수 있으며, 관련 이용 경로와 URL은 문자로 안내해 드리겠습니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=환전 문의 
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3945
과거 질의응답:
요구사항:외화 통장의 현금 인출 가능 여부와 수수료,  온라인 절차를 안내하시오.
고객질문:외화 통장에서 보유 중인 달러를 현금으로 직접 인출할 수 있습니까?
상담사답변:네, 현재 공항 지점에서도 해당 계좌를 조회하고 현금 인출이 가능하지만, 외화 현금 출금은 지점에 직접 방문하셔야 하며 사전 예약은 필요하지 않습니다. 인출 시에는 본인 신분증을 지참하고 영업시간인 오전 9시부터 오후 4시 사이에 방문하셔야 하며, 인출 금액에 1.0%의 현찰수수료가 자동으로 차감됩니다. 수수료 누적에 대해 고민이시라면, 저희 앱에서 프로 우대를 적용받아 공항 환전소에서 직접 수령하실 수 있는 방법이 있습니다.
꼬리질문:온라인으로도 가능하면 편리할 것 같아요. 절차가 어떻게 되는지 알려 주세요.
종합답변:외화 통장에 보유하신 달러는 지점을 방문하시면 신분증 확인 후 현금으로 인출하실 수 있으며, 별도 예약 없이 오전 9시부터 오후 4시까지 가능합니다. 인출 시 1.0%의 현찰 수수료가 자동으로 차감되며, 모바일 앱을 통한 온라인 환전은 실시간 환율이 적용되어 원화 계좌로 입금되며 수수료는 발생하지 않으나 하루 최대 이체 한도와 신분증 인증이 필요합니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=외화 현금 인출
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.2968
과거 질의응답:
요구사항:외화 통장의 현금 인출 가능 여부와 수수료,  온라인 절차를 안내하시오.
고객질문:외화 통장에서 보유 중인 달러를 현금으로 직접 인출할 수 있습니까?
상담사답변:네, 현재 공항 지점에서도 해당 계좌를 조회하고 현금 인출이 가능하지만, 외화 현금 출금은 지점에 직접 방문하셔야 하며 사전 예약은 필요하지 않습니다. 인출 시에는 본인 신분증을 지참하고 영업시간인 오전 9시부터 오후 4시 사이에 방문하셔야 하며, 인출 금액에 1.0%의 현찰수수료가 자동으로 차감됩니다. 수수료 누적에 대해 고민이시라면, 저희 앱에서 프로 우대를 적용받아 공항 환전소에서 직접 수령하실 수 있는 방법이 있습니다.
꼬리질문:온라인으로도 가능하면 편리할 것 같아요. 절차가 어떻게 되는지 알려 주세요.
종합답변:외화 통장에 보유하신 달러는 지점을 방문하시면 신분증 확인 후 현금으로 인출하실 수 있으며, 별도 예약 없이 오전 9시부터 오후 4시까지 가능합니다. 인출 시 1.0%의 현찰 수수료가 자동으로 차감되며, 모바일 앱을 통한 온라인 환전은 실시간 환율이 적용되어 원화 계좌로 입금되며 수수료는 발생하지 않으나 하루 최대 이체 한도와 신분증 인증이 필요합니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=외화 현금 인출
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3385
과거 질의응답:
요구사항:해외 환전 현금 수령 가능 여부와 적용되는 수수료, 공항환전 절차를 안내하시오.
고객질문:환전을 현금으로 받고 싶습니다. 직접 지점에 가서 체크카드만 있으면 받을 수 있는지 ,수수료,  국내에서 현금을 받는 방법에 대해서 궁금합니다.
상담사답변:현금 수령을 원하실 경우, 일반적인 환전 서비스는 해외 현지에서만 가능하고, 국내에서는 전자 형태로만 제공됩니다.수수료는 환전 금액의 0.5%가 부과되며, 추가로 현금 인출 시 해외 ATM 이용 시 발생하는 현지 수수료가 별도로 청구될 수 있음을 알려드립니다.외화 머니로, 백프로 환율 우대를 받으면서 충전한 경우 국내에서 출금이나 수령이 불가능합니다. 발급받은 ★★ 카드를 해외 현지에서 사용해 등기소에서 인출하거나 결제 시 충전됩니다. 
꼬리질문:공항에서 환전 후 현금으로 받은 적이 있는데, 그 절차가 어떻게 되나요?
종합답변:고객님께서는 현금 수령 가능 여부와 체크카드 이용에 대해 문의하셨습니다. 국내에서는 외화 환전이 전자 방식으로만 가능하며 현금 수령은 해외 현지에서만 가능하니 참고해 주시기 바랍니다. 환전 시 0.5% 수수료가 부과되며 해외 ATM 이용 시 추가 현지 수수료가 발생할 수 있습니다. 공항에서 환전하실 경우 앱에서 통화를 선택하고 미리 예약하시면 지정한 수령점에서 신분증을 제시하여 현금을 받으실 수 있으며, 관련 이용 경로와 URL은 문자로 안내해 드리겠습니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=환전 문의 
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3538
과거 질의응답:
요구사항:해외 현금인출 관련 수수료 및 한도 안내를 제공하시오.
고객질문:무료 환전으로 ★★에서 1,000달러를 환전했는데, 해외에서 현금이 필요합니다. 카드로 현금 인출이 가능한지, 수수료는 어떻게 되는지, 그리고 현지에서 현찰을 받는 방법을 알려 주세요.
상담사답변:해외에서 현금 인출 시 이용 가능한 ATM 네트워크와 한도, 그리고 현지 통화로 자동 전환되는 방식에 대해 안내드립니다. 또한 인출 시 적용되는 시디기 이용 관리비 수수료와 우대율에 대한 구체적인 계산 예시를 알려드리겠습니다.
꼬리질문:인출 한도는 어떻게 되나요? 그리고 수수료는 정확히 얼마인지 예시를 들어 주세요.
종합답변:해외 현금 인출은 카드 소지 후 현지 ATM을 이용하면 가능합니다. 사용 가능한 ATM 네트워크는 주요 국제 브랜드이며, 인출 시 현지 통화로 자동 전환됩니다. 인출에 적용되는 시디기 이용 관리비 수수료는 인출 금액의 1%이며 최소 5달러가 부과됩니다. 우대율이 적용될 경우 수수료 일부가 할인될 수 있습니다. 인출 한도는 하루 최대 500달러이며, 월 누적 한도는 2,000달러까지 가능합니다. 예를 들어 100달러를 인출하면 수수료는 1달러이며, 최소 금액보다 작을 경우에도 5달러가 부과됩니다. 우대율 적용 시 이 금액이 일부 차감될 수 있습니다.
metadata:
- consulting_category=은행
- consulting_topic=환전문의
- qa_topic=환전문의
- consulting_purpose=해외 현금인출 안내
---------------------------------------------------
'''

In [ ]:
# 증권 - 하루에 모바일 OTP 생성은 몇 번까지 가능한가요? (21-1_sec_01_012766)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 하루에 모바일 OTP 생성은 몇 번까지 가능한가요?


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3080
과거 질의응답:
요구사항:모바일 OTP 발급 절차와 일일 한도, 건당 한도, 이체 수수료 및 이체확인 절차를 확인하여 안내하시오.
고객질문:모바일 OTP를 발급받으려면 어떤 절차가 필요한가요? 일일 한도와 건당 한도가 어떻게 되나요? 이체 수수료는 얼마나 되나요?
상담사답변:모바일 OTP는 앱을 통해 본인 인증 후 발급되며, 인증 절차는 휴대폰 번호 인증, 신분증 사진 촬영, 보안 질문 응답으로 이루어집니다. 사용 중이신 휴대폰 번호로 모바일 OTP 발급 안내 링크를 문자로 보내드리겠습니다. 링크를 클릭하시고 안내에 따라 본인 인증 및 신분증 촬영을 완료해 주시고 절차를 진행해 주시면, 발급이 완료된 후 바로 이체에 이용하실 수 있습니다. 모바일 OTP 기준으로는 한 건당 최대 ●●●●●원, 하루 최대 ●●●●●원까지 이체가 가능합니다. 이체 수수료는 이체 금액의 0.1%이며, 최소 ●●●●원부터 부과됩니다. 
꼬리질문:이체 후에 확인 메시지는 어떻게 받나요?
종합답변:모바일 OTP는 휴대폰 번호 인증, 신분증 촬영, 보안 질문 응답을 통해 본인 인증 후 발급됩니다. 발급 안내 링크를 문자로 보내드리니, 안내에 따라 인증 절차를 완료하시면 바로 이체에 이용하실 수 있으며, 이체 시 수수료와 한도에 대한 안내도 함께 확인하실 수 있습니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=자금이체/계좌제한
- consulting_purpose=모바일 OTP 발급
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3393
과거 질의응답:
요구사항:OTP 재설정 가능 횟수와 초과 시 조치 방법을 안내하시오.
고객질문:OTP 재설정은 하루에 몇 번까지 가능한지 궁금합니다. 만약 재설정 횟수 제한을 초과하게 되면 어떤 절차를 거쳐야 하는지 자세히 알려주실 수 있나요?
상담사답변:OTP 재설정은 하루 최대 3회까지 가능하도록 제한되어 있으며, 이 횟수를 초과할 경우 고객님의 보안 등급에 따라 추가적인 인증 절차가 요구될 수 있습니다. 만약 더 많은 지원이 필요하시면 언제든지 콜센터로 연락 주시면 별도의 안내와 도움을 받으실 수 있습니다.
꼬리질문:만약 추가적인 보안 인증이나 보정이 필요한 상황이 발생하면, 그 절차는 어떻게 진행해야 하는지 구체적으로 설명해 주실 수 있나요?
종합답변:OTP는 하루에 최대 3회까지 재설정할 수 있습니다. 이 횟수를 초과하면 보안 등급에 따라 추가 인증 절차가 요구되며, 경우에 따라 콜센터에 연락하여 별도 지원을 받아야 합니다. 보정이 추가로 필요할 경우, 로그인 후 앱 상단 메뉴에서 ‘보정 요청’ 항목을 선택하고 안내에 따라 진행하시기 바랍니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=재설정 제한
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3608
과거 질의응답:
요구사항:OTP 재발급 가능 횟수와 수수료 안내를 하시오.
고객질문:OTP를 재발급받을 때 하루에 몇 번까지 신청할 수 있는지 궁금합니다. 그리고 재발급 과정에서 별도의 수수료가 발생하는지도 자세히 알려 주실 수 있을까요?
상담사답변:보안 정책에 따라 OTP 재발급은 하루 최대 3회까지 가능하며, 재발급할 때마다 2,000원의 소액 수수료가 부과됩니다. 미리 발급받아 두시면 긴급 상황에서 보다 편리하게 이용하실 수 있으니 참고해 주시기 바랍니다.
꼬리질문:만약 하루에 3회를 초과해서 재발급을 신청해야 하는 특별한 상황이 생기면 어떻게 해야 하나요? 추가 절차나 문의할 곳이 있는지도 알려 주세요.
종합답변:OTP는 보안 정책에 따라 하루 최대 3회까지 재발급이 가능하며, 한 건당 2,000원의 수수료가 부과됩니다. 급한 상황에 대비해 미리 발급받아 두시는 것이 좋습니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=OTP 재발급 제한
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3259
과거 질의응답:
요구사항:모바일 OTP 재발급 비용과 일일 이용 횟수를 안내하시오.
고객질문:모바일 OTP를 다시 발급받을 때 별도의 비용이 발생하는지 궁금한데, 혹시 재발급 수수료가 따로 부과되는지 알려주실 수 있나요?
상담사답변:모바일 OTP 재발급과 관련해서는 현재 별도의 수수료가 전혀 부과되지 않고 있으며, 기존에 사용하시던 인증 수단과 동일하게 무료로 제공되고 있으니 안심하셔도 됩니다. 또한 하루에 최대 5회까지 OTP 인증 시도를 하실 수 있고, 만약 이 횟수를 초과하면 다음 날까지 인증 시도가 제한되는 점 참고 부탁드립니다.
꼬리질문:그렇다면 하루 동안 OTP 인증을 몇 번까지 사용할 수 있는지 정확하게 알려주실 수 있을까요?
종합답변:모바일 OTP 재발급은 비용이 전혀 발생하지 않으며 무료로 제공됩니다. 추가로, OTP 인증은 하루에 최대 5회까지 사용 가능하고, 이를 초과하면 다음 날까지 제한이 해제됩니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=재발급 비용 확인
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3930
과거 질의응답:
요구사항:모바일 OTP 발급 절차와 일일 한도, 건당 한도, 이체 수수료 및 이체확인 절차를 확인하여 안내하시오.
고객질문:모바일 OTP를 발급받으려면 어떤 절차가 필요한가요? 일일 한도와 건당 한도가 어떻게 되나요? 이체 수수료는 얼마나 되나요?
상담사답변:모바일 OTP는 앱을 통해 본인 인증 후 발급되며, 인증 절차는 휴대폰 번호 인증, 신분증 사진 촬영, 보안 질문 응답으로 이루어집니다. 사용 중이신 휴대폰 번호로 모바일 OTP 발급 안내 링크를 문자로 보내드리겠습니다. 링크를 클릭하시고 안내에 따라 본인 인증 및 신분증 촬영을 완료해 주시고 절차를 진행해 주시면, 발급이 완료된 후 바로 이체에 이용하실 수 있습니다. 모바일 OTP 기준으로는 한 건당 최대 ●●●●●원, 하루 최대 ●●●●●원까지 이체가 가능합니다. 이체 수수료는 이체 금액의 0.1%이며, 최소 ●●●●원부터 부과됩니다. 
꼬리질문:이체 후에 확인 메시지는 어떻게 받나요?
종합답변:모바일 OTP는 휴대폰 번호 인증, 신분증 촬영, 보안 질문 응답을 통해 본인 인증 후 발급됩니다. 발급 안내 링크를 문자로 보내드리니, 안내에 따라 인증 절차를 완료하시면 바로 이체에 이용하실 수 있으며, 이체 시 수수료와 한도에 대한 안내도 함께 확인하실 수 있습니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=자금이체/계좌제한
- consulting_purpose=모바일 OTP 발급
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3997
과거 질의응답:
요구사항:모바일 OTP 사용 절차와 인증번호 입력 시간 제한을 설명하시오.
고객질문:새로운 보안 매체로 모바일 OTP를 등록했습니다. 상담사님 말씀처럼 실행 버튼을 눌러 비밀번호와 인증번호를 입력해야 한다고 하셨는데, 실제로 어떤 절차로 OTP를 활성화하고 사용할 수 있는지 알려 주세요.
상담사답변:모바일 OTP는 앱 메뉴에서 인증센터로 들어가 실행 버튼을 눌러 활성화합니다. 이후 여섯 자리 비밀번호를 입력하면 인증번호 요청 버튼이 활성화됩니다. 발급된 인증번호 여섯 자리를 HTS나 앱 화면에 입력하면 인증이 완료됩니다. 앱에서만 사용할 경우 실행만으로도 바로 이용할 수 있습니다.
꼬리질문:OTP 인증번호는 몇 초 안에 입력해야 하나요? 글자수: 72
종합답변:모바일 OTP는 실행 버튼을 누른 후 비밀번호를 입력하면 인증번호가 발급되며, 이 번호는 30초 이내에 입력해야 만료되지 않습니다. 인증번호 입력을 여러 번 실패하면 일시적으로 잠금이 발생할 수 있으니 빠르게 입력해 주시기 바랍니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=보안 인증
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3232
과거 질의응답:
요구사항:모바일 OTP 재발급 비용과 일일 이용 횟수를 안내하시오.
고객질문:모바일 OTP를 다시 발급받을 때 별도의 비용이 발생하는지 궁금한데, 혹시 재발급 수수료가 따로 부과되는지 알려주실 수 있나요?
상담사답변:모바일 OTP 재발급과 관련해서는 현재 별도의 수수료가 전혀 부과되지 않고 있으며, 기존에 사용하시던 인증 수단과 동일하게 무료로 제공되고 있으니 안심하셔도 됩니다. 또한 하루에 최대 5회까지 OTP 인증 시도를 하실 수 있고, 만약 이 횟수를 초과하면 다음 날까지 인증 시도가 제한되는 점 참고 부탁드립니다.
꼬리질문:그렇다면 하루 동안 OTP 인증을 몇 번까지 사용할 수 있는지 정확하게 알려주실 수 있을까요?
종합답변:모바일 OTP 재발급은 비용이 전혀 발생하지 않으며 무료로 제공됩니다. 추가로, OTP 인증은 하루에 최대 5회까지 사용 가능하고, 이를 초과하면 다음 날까지 제한이 해제됩니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=재발급 비용 확인
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3601
과거 질의응답:
요구사항:OTP 재설정 가능 횟수와 초과 시 조치 방법을 안내하시오.
고객질문:OTP 재설정은 하루에 몇 번까지 가능한지 궁금합니다. 만약 재설정 횟수 제한을 초과하게 되면 어떤 절차를 거쳐야 하는지 자세히 알려주실 수 있나요?
상담사답변:OTP 재설정은 하루 최대 3회까지 가능하도록 제한되어 있으며, 이 횟수를 초과할 경우 고객님의 보안 등급에 따라 추가적인 인증 절차가 요구될 수 있습니다. 만약 더 많은 지원이 필요하시면 언제든지 콜센터로 연락 주시면 별도의 안내와 도움을 받으실 수 있습니다.
꼬리질문:만약 추가적인 보안 인증이나 보정이 필요한 상황이 발생하면, 그 절차는 어떻게 진행해야 하는지 구체적으로 설명해 주실 수 있나요?
종합답변:OTP는 하루에 최대 3회까지 재설정할 수 있습니다. 이 횟수를 초과하면 보안 등급에 따라 추가 인증 절차가 요구되며, 경우에 따라 콜센터에 연락하여 별도 지원을 받아야 합니다. 보정이 추가로 필요할 경우, 로그인 후 앱 상단 메뉴에서 ‘보정 요청’ 항목을 선택하고 안내에 따라 진행하시기 바랍니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=재설정 제한
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3690
과거 질의응답:
요구사항:고객의 OTP 발급 절차와 필요한 인증 방법을 안내하시오.
고객질문:오랜만에 로그인을 했는데 모바일 OTP를 발급받으려니 OTP 번호를 입력하라는 화면이 나왔습니다. 옛날에 실물 OTP를 등록했었던 것 같습니다. 어떻게 해야 할까요?
상담사답변:OTP 발급 과정에서 화면이 바뀌어 불편을 겪으시는 점 충분히 이해합니다. 현재 실물 OTP가 없으시다면 새로운 인증 방법을 안내해 드리겠습니다. 먼저 사용 중인 휴대폰이 본인 명의인지 확인해 주시고, 기기가 스마트폰인지 알려 주세요. 아이디는 기억하지만 비밀번호가 없으신 경우, 계좌 비밀번호 네 자리를 이용해 인증할 수 있습니다. ★★ 인증(본인 명의 휴대폰 문자)과 계좌 비밀번호 두 가지 인증으로 등록이 가능하니, 인증센터 메뉴에서 ★★ 재등록을 진행해 주세요. 문자 인증까지 완료하면 계좌번호가 조회되고 네 자리 비밀번호를 입력한 뒤 여섯 자리 비밀번호를 설정합니다. 이후 새로 만든 ★★ 비밀번호로 로그인하고 기존 OTP를 해지한 뒤 모바일 OTP를 발급받으시면 됩니다.
꼬리질문:모바일 OTP 사용 시 일일 이체 한도가 1천만 원이라고 들었는데, 실제로 사용 가능한 금액이 제한되나요?
종합답변:먼저 고객님의 실물 OTP가 없음을 확인하고, 본인 명의 휴대폰과 스마트폰 여부를 확인합니다. 아이디는 기억하지만 비밀번호가 없으신 경우, 계좌 비밀번호 네 자리를 이용해 인증합니다. ★★ 인증(본인 명의 휴대폰 문자)과 계좌 비밀번호 두 가지 인증으로 신규 인증을 진행합니다. 인증센터에서 ★★ 재등록을 선택하고, 문자 인증을 완료하면 계좌번호가 조회됩니다. 이후 네 자리 비밀번호를 입력하고 여섯 자리 비밀번호를 설정합니다. 새로 만든 ★★ 비밀번호로 로그인한 뒤 기존 OTP를 해지하고 모바일 OTP를 발급받으시면 됩니다. 모바일 OTP의 일일 이체 한도는 기본 1천만 원이며, 특정 조건을 충족하면 상향 조정이 가능합니다.
metadata:
- consulting_category=증권
- consulting_topic=HTS/MTS
- qa_topic=HTS/MTS
- consulting_purpose=OTP 발급
---------------------------------------------------
'''

In [ ]:
# 증권 - 성년 후견인 신청을 한 상태인데 계좌 해지를 하고 싶어요. 이때 필요한 서류를 알려주세요. (21-1_sec_07_014014)
user_query = input("질문을 입력하세요: ")
print(f"사용자 질의: {user_query}\n")

for col in ["embedding_q", "embedding_i", "embedding_full"]:
    print(f"\n{'='*60}")
    print(f"[{col}] 기준 검색 결과")
    print(f"{'='*60}")
    results = search_top3(user_query, embedding_column=col)
    print_results(results)

In [ ]:
'''
사용자 질의: 성년 후견인 신청을 한 상태인데 계좌 해지를 하고 싶어요. 이때 필요한 서류를 알려주세요.


============================================================
[embedding_q] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3515
과거 질의응답:
요구사항:자동이체 해지 절차와 필요 서류를 안내하시오.
고객질문:안녕하세요, 계좌를 해지하려고 하는데 구체적으로 어떤 방법들이 있는지 그리고 해지할 때 꼭 준비해야 하는 서류에는 어떤 것들이 포함되는지 자세하게 알려주실 수 있을까요?
상담사답변:계좌 해지는 모바일 뱅킹 앱을 통해서도 간편하게 진행하실 수 있으며, 직접 가까운 영업점을 방문하시거나 고객센터에 전화로 신청하는 방법도 모두 가능합니다. 이 과정에서 본인 확인을 위해 신분증과 함께 추가로 필요한 서류를 반드시 준비해 주셔야 원활한 처리가 이루어집니다.
꼬리질문:그렇군요, 그러면 모바일 뱅킹을 이용해서 직접 계좌를 해지하는 구체적인 절차와 단계별 방법을 상세하게 안내해 주실 수 있으신가요?
종합답변:자동이체 해지는 모바일 뱅킹 앱에서 ‘자동이체 관리’ 메뉴를 선택하신 후 해지하려는 계좌를 찾아 ‘해지’ 버튼을 누르면 됩니다. 해지 절차 중에 인증 문자가 발송되며, 해당 문자의 6자리 코드를 입력하면 해지가 완료됩니다. 필요 서류는 신분증 및 본인 확인용 서류이며, 영업점 방문 시 지참해 주시면 됩니다.
metadata:
- consulting_category=은행
- consulting_topic=이자/연체금액
- qa_topic=이자/연체금액
- consulting_purpose=자동이체 해지 안내
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3571
과거 질의응답:
요구사항:계좌 해지 절차와 잔액 확인, 수수료 및 ISA 원금 반환에 대해 상세히 안내하시오.
고객질문:계좌 해지와 관련된 절차와 필요 서류에 대해 문의드립니다.
상담사답변:해지 요청이시군요. 먼저 고객님의 계좌를 확인하고 진행 절차를 안내해 드리겠습니다. 성함을 알려주시면 비밀번호 네 자리와 우물 정자 정보를 입력해 주시기 바랍니다. 확인이 완료되면 해지 진행 상황을 문자로 안내해 드리며, 잔액이 남아 있는 경우 전액 출금 후에만 계좌 폐쇄가 가능함을 알려드립니다.
꼬리질문:현재 계좌 잔고가 얼마인지 확인하고 싶습니다. 또한 ISA 계좌 개설 시 입금된 원금이 언제 반환되는지, 해지 시 추가 수수료가 발생하는지 알려주시겠습니까?
종합답변:고객님께서는 계좌 해지를 원하십니다. 먼저 성함과 비밀번호 네 자리, 우물 정자 정보를 제공해 주시면 계좌 확인이 가능합니다. 확인이 완료되면 현재 잔액이 1,200,000원임을 알려드립니다. 잔액이 남아 있는 경우에는 전액 출금 후에만 계좌 폐쇄가 가능하므로, 전액 출금을 요청하시고 출금이 완료되면 문자로 진행 상황을 안내해 드리겠습니다. 출금 요청 후 영업일 기준 1일 이내에 지정하신 계좌로 이체됩니다. 이체 시 발생할 수 있는 은행 이체 수수료는 최소 금액이 적용되며, 해지 자체에 추가 수수료는 없습니다. ISA 계좌 개설 시 입금된 원금도 출금 요청 후 영업일 기준 1일 이내에 지정 계좌로 반환됩니다. 모든 절차가 완료되면 계좌 폐쇄를 진행하고, 최종 확인 문자와 함께 안내드리겠습니다.
metadata:
- consulting_category=증권
- consulting_topic=자금이체/계좌제한
- qa_topic=자금이체/계좌제한
- consulting_purpose=계좌 해지
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3574
과거 질의응답:
요구사항:계좌해지 시 주의사항 안내 및 계좌 연동 서비스 비활성 여부에 대해 안내하시오.
고객질문:계좌 해지 절차를 진행하면서 혹시 추가로 확인해야 할 서류나 주의사항이 있나요? 예를 들어, 잔액이 남아 있으면 어떻게 해야 하는지도 알려 주세요.
상담사답변:해지 전에는 계좌 잔액이 0원인지 반드시 확인해 주시고, 남은 금액이 있을 경우 다른 계좌로 이체하시거나 현금으로 인출하셔야 합니다. 또한, 해지 신청 후에는 잔액이 없음을 확인하는 최종 확인 문자를 받게 됩니다.
꼬리질문: 계좌 해지 이후에도 기존에 사용하던 카드나 체크 서비스는 별도의 신청없이 자동으로 비활성화되는지 궁금합니다.
종합답변:계좌 해지 전에는 잔액이 0원인지 꼭 확인하시고, 남은 금액이 있다면 다른 계좌로 이체하시거나 현금으로 인출해 주시기 바랍니다. 해지 후에는 잔액이 없음을 확인하는 최종 문자를 받게 되며, 연동된 체크카드와 신용카드는 자동으로 사용이 중지되나 카드 사용 내역이 남아 있다면 해당 카드사에 문의해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=계좌해지시 주의사항과 연동서비스 비활성여부 문의
---------------------------------------------------

============================================================
[embedding_i] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3463
과거 질의응답:
요구사항:계좌 해지 시 필요 서류 및 신청 방법을 안내하시오.
고객질문:해지에 필요한 서류가 정확히 무엇인지 알고 싶습니다. 신분증 외에 추가로 제출해야 할 서류가 있나요? 자동이체 등록 목적이 없다고 하면 서류 제출이 더 필요하나요? 자동이체가 없을 경우 해지 절차가 달라지는지, 그리고 어떤 서류를 추가로 제출해야 하는지 구체적으로 설명해 주세요. 
상담사답변:해지 시 기본적으로 신분증이 필요하고, 실업 급여를 수령 중이시라면 최근 3개월 이내 급여 수령 확인서 또는 고용보험 수급 확인서가 추가로 요구될 수 있습니다. 또한, 계좌에 남은 금액이 있을 경우 정산 내역서도 준비해 주시면 원활합니다. 자동이체 등록 목적이 없으시다면 별도의 추가 서류는 필요하지 않으며, 기존 자동이체가 설정되어 있지 않은지 확인만 하면 됩니다. 자동이체가 존재한다면 해지 전 해제 신청을 먼저 진행해 주셔야 합니다.
꼬리질문:영업점을 직접 방문하지 않고 우편이나 온라인으로 해지를 신청할 수 있는 방법이 있는지 궁금합니다. 가능하다면 필요한 서류와 절차를 단계별로 알려 주시고, 우편 발송 시 주의사항도 알려 주세요.
종합답변:계좌 해지 시에는 신분증이 필요하며, 실업 급여를 받고 계신 경우 최근 3개월 이내 급여 수령 확인서나 고용보험 수급 확인서가 추가로 필요할 수 있습니다. 자동이체가 설정되어 있다면 해지 전에 해제 신청을 하셔야 하며, 온라인 해지 신청은 불가능하니 서류 원본과 서명 인증을 포함한 우편 신청을 지정된 주소로 보내 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=금융거래한도/비대면한도계좌
- qa_topic=만기,연장/해지,수신
- consulting_purpose=계좌 해지 서류 문의
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3517
과거 질의응답:
요구사항:계좌 해지 절차와 필요한 서류 및 조건을 안내하시오.
고객질문:죄송한데요, 계좌 해지를 하는 거예요. 이제 해지가 안돼가지구요.
상담사답변:혹시 대상이 아닙니다라고 나오신 겁니까? 현재 화면에 표시된 문구가 정확히 그렇게 나오고 있는지 확인 부탁드립니다. 부분 한도 제한 해제는 보통 급여를 ★★은행으로 받고 계시거나 그런 경우에 해당됩니다. 강 보험 자격 득실 확인서는 검증을 통해 이루어지는 부분이기 때문에요. 대상이 아닙니다라고 나오셨다면, 혹시 급여를 ★★은행으로 받고 계십니까? 이 부분을 자세히 여쭤보는 이유는 한도 해제 기준과 직접적인 연관이 있기 때문입니다. 처음 급여를 받으셨다고 말씀하셨는데, 정확히 언제부터 해당 계좌에 입금이 시작되었는지 확인해 주실 수 있으실까요? 입금 일자를 파악하면 심사 진행에 도움이 됩니다. 그럼 현재 말씀하신 계좌에 출금 제한이 ●●●원으로 설정되어 있다는 말씀이시죠? 제한 금액과 관련된 추가적인 확인이 필요합니다. 그런 부분이 아니시라면 ●●월 ●에 이렇게 필요하십니다. 해당 시점에 필요한 절차와 서류에 대해 미리 안내드리겠습니다. 세 번 급여를 받으셔야 기본적으로 한도 제한 해제 심사를 진행할 수 있는 대상이 되기 때문입니다. 급여 실적이 누적될 경우 심사 결과가 긍정적으로 나올 가능성이 높습니다. 신입으로 이체하셔서 전화를 받으신 게 아니시라면 번거로우시겠지만 세 번에 급여 실적이 필요합니다. 실적이 확인되면 한도 제한 해제가 원활히 진행됩니다. 현재 고객님께서 제공해 주실 수 있는 급여 입금 내역이 없으신 경우, 다른 소득 증빙 서류를 대신 제출하실 수 있습니다. 예를 들어 사업소득 증명서 등이 해당됩니다. 네, 세 번 맞을 통한 에는 번거로우시겠지만 모바일이나 인터넷뱅킹에서 하루에 ●●원 출금 이체 가능하시구요. 이와 동시에 온라인으로 신청서를 작성하실 수 있습니다. ★★★★ 기기에서도 ●●원 출금이 가능하도록 설정을 변경하실 수 있습니다. 기기별 제한 해제 방법을 자세히 안내드리겠습니다. 창구 직원 통해서 ●● ●원 이렇게 가능하십니다. 직접 방문하셔서 직원에게 요청하시면 즉시 처리가 가능합니다. 네, 그래서 이 부분이 너무 좀 오래 걸린다라고 생각이 되시면 번거로우시겠지만 가까운 ★★은행 영업점에 방문하셔서 혹은 다른 방법으로 빠르게 한도 제한 해제가 가능한지 상담을 먼저 받아 보셔야 됩니다. 방문 전 필요한 서류 목록을 미리 알려드리겠습니다. 네, 맞습니다. 내일 예정된 지급액이 반영되면 자동으로 한도 제한이 조정될 수도 있습니다. 추가로 궁금하신 점이 있으면 언제든지 전화 주시거나 온라인 채팅을 이용해 주세요. 고객님의 편의를 위해 24시간 상담이 가능합니다.
꼬리질문:지금 바로 통화가 끝 나는 건가요.
종합답변:계좌 해지가 현재 진행되지 않는 경우, 화면에 표시되는 문구가 "대상이 아닙니다"인지 먼저 확인해 주시기 바랍니다. 한도 제한 해제는 급여를 특정 은행(예: ★★은행)으로 받는 경우에 적용될 수 있으며, 급여 입금 실적이 필요합니다. 급여 입금이 시작된 일자를 알려 주시면 심사에 도움이 됩니다. 현재 계좌에 설정된 출금 제한 금액도 확인이 필요합니다. 만약 급여 입금 내역이 없으시다면, 사업소득 증명서와 같은 다른 소득 증빙 서류를 제출하실 수 있습니다. 제한 해제 절차는 모바일·인터넷뱅킹에서 하루에 일정 금액(●●원)까지 출금이 가능하도록 설정하거나, 영업점 방문을 통해 직접 요청하실 수 있습니다. 방문 전 필요한 서류 목록을 미리 안내해 드리며, 내일 예정된 급여가 입금되면 자동으로 제한이 조정될 수도 있습니다. 추가 문의 사항이 있으면 언제든 전화나 온라인 채팅으로 연락해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=금융거래한도/비대면한도계좌
- qa_topic=금융거래한도/비대면한도계좌
- consulting_purpose=계좌 해지 절차
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3534
과거 질의응답:
요구사항:계좌 해지 절차와 필요 서류를 안내하시오.
고객질문:제가 보유하고 있는 그 계좌를 해지하게 되면 어떤 절차가 진행되는지 궁금합니다. 특히 해지 과정에서 필요한 단계와 함께 계좌에 남아 있는 잔액은 어떻게 처리되는지 자세히 알려주실 수 있나요?
상담사답변:계좌 해지는 가까운 지점을 직접 방문하시거나 ★★은행 모바일 어플리케이션을 통해서도 간편하게 진행하실 수 있습니다. 해지 후 남아 있는 잔액은 고객님께서 원하시는 다른 계좌로 안전하게 이체하거나, 현금으로 수령하실 수 있도록 안내해 드리고 있습니다. 지점 방문 시에는 신분증과 해당 계좌 관련 서류를 반드시 지참하시면 절차가 더욱 원활하게 진행됩니다.
꼬리질문:그러면 영업점에 직접 방문해서도 계좌 해지가 가능한가요? 방문 전에 꼭 준비해야 할 서류나 필요한 사항이 있다면 미리 알려주시면 감사하겠습니다.
종합답변:계좌를 해지하시려면 영업점 방문 또는 모바일 앱을 이용하실 수 있습니다. 영업점 방문 시 신분증과 계좌 관련 서류(예: 통장, 서명 카드)를 지참하시면 절차가 원활합니다. 앱에서는 메뉴 선택만으로 간편하게 해지 가능하며, 해지 후 남은 금액은 다른 계좌로 이체하거나 현금으로 수령하실 수 있습니다. 잔액이 0원이 된 후에 해지가 완료됩니다.
metadata:
- consulting_category=은행
- consulting_topic=금융거래한도/비대면한도계좌
- qa_topic=금융거래한도/비대면한도계좌
- consulting_purpose=계좌 해지 절차
---------------------------------------------------

============================================================
[embedding_full] 기준 검색 결과
============================================================
---------------------------------------------------
[1순위] 거리: 0.3625
과거 질의응답:
요구사항:미성년자 계좌 해지 절차와 필요 서류, 영업점 방문 조건 및 해지 후 잔액 처리 방식을 안내하시오.
고객질문:자녀 명의의 OO은행 청약 통장을 해지하려고 합니다. 미성년자 계좌는 어떻게 해지할 수 있고, 어떤 서류가 필요한가요?
상담사답변:미성년자 명의의 계좌를 해지하려면 법정대리인(부모 또는 보호자)이 OO은행 영업점을 방문하여 신청하셔야 합니다. 해지 시에는 자녀의 신분증(주민등록등본 또는 가족관계증명서 포함), 법정대리인의 신분증, 계좌번호, 비밀번호 등이 필요하며, 일부 상품은 해지 신청서 작성이 요구됩니다. 청약 통장 해지 시에는 이자 및 세금 처리 조건도 함께 안내되며, 해지 후 잔액은 지정 계좌로 이체됩니다. 모바일 앱이나 인터넷뱅킹에서는 미성년자 계좌 해지가 불가능하므로 반드시 영업점에서 직접 처리하셔야 합니다.


꼬리질문:청약 통장 해지 후 재가입은 가능한가요? 해지한 계좌의 이자는 어떻게 처리되나요?
종합답변:미성년자 계좌 해지는 법정대리인이 은행 영업점을 방문하여 신청하셔야 하며, 자녀와 법정대리인의 신분증, 계좌번호, 비밀번호가 필요합니다. 해지 후 잔액은 지정한 계좌로 이체되며, 청약 통장의 경우 이자와 세금 처리에 따라 일부 금액이 조정될 수 있으니 참고해 주시기 바랍니다.
metadata:
- consulting_category=은행
- consulting_topic=이자/연체금액
- qa_topic=이자/연체금액
- consulting_purpose=미성년자 계좌 해지 절차
---------------------------------------------------
---------------------------------------------------
[2순위] 거리: 0.3869
과거 질의응답:
요구사항:계좌 폐쇄 절차와 필요 서류를 안내하시오.
고객질문:미성년 자녀 계좌 해지하려구요.
상담사답변:ARS 연결하면 비밀번호 네 자리와 우편번호를 전화기로 입력 부탁드립니다. 은행 계좌 폐쇄는 해당 은행에 직접 요청하셔야 합니다. 은행 영업시간에 따라 처리되며 보통 1~3일 정도 소요됩니다. 우선 증권 계좌 폐쇄 처리는 완료했으며 전자금융 약정도 해지되었습니다. 이제 모바일이나 PC 접근이 차단됩니다. 전자금융 약정 해지와 아이디 삭제를 원하시는 것이 맞습니까? 전자금융 약정 해지와 회원 탈퇴에 해당하는 아이디 삭제를 진행 중입니다. 광고성 마케팅 수신을 원치 않으시겠습니까? 모든 계좌 폐쇄가 완료되었습니다. 추가로 도와드릴 사항이 있으면 알려 주세요. 해지 완료 후 계좌에 남은 현금은 지정하신 계좌로 이체해 드립니다. 이체 계좌를 확인해 주시겠어요? 청약증서나 세금계산서가 필요하시면 발송해 드릴 수 있습니다.
꼬리질문:혹시 하나 지 ★★ 계좌도 폐지 가능 되나요?
종합답변:먼저, 미성년 자녀 계좌를 해지하시려면 ARS에 연결하여 비밀번호 네 자리와 우편번호를 입력해 주시기 바랍니다. 입력이 확인되면 해당 계좌는 은행에 직접 폐쇄 요청이 진행됩니다. 은행 영업시간에 따라 처리되며 보통 1~3일 정도 소요됩니다. 동시에 증권 계좌 폐쇄와 전자금융 약정 해지가 완료되어 모바일 및 PC 접근이 차단됩니다. 전자금융 약정 해지와 함께 아이디 삭제를 원하시는 경우, 추가 확인 후 진행됩니다. 또한 광고성 마케팅 수신을 원치 않으시는 경우 해당 설정도 함께 해드리겠습니다. 모든 계좌 폐쇄가 완료된 후, 남은 현금은 고객님이 지정하신 계좌로 이체됩니다. 이체 계좌를 확인해 주시고, 필요하신 경우 청약증서나 세금계산서를 발송해 드리겠습니다. 추가로 다른 ★★ 계좌도 폐지 가능하오니, 해당 계좌 번호를 알려 주시면 동일 절차로 진행해 드리겠습니다.
metadata:
- consulting_category=증권
- consulting_topic=자금이체/계좌제한
- qa_topic=자금이체/계좌제한
- consulting_purpose=계좌 폐쇄 요청
---------------------------------------------------
---------------------------------------------------
[3순위] 거리: 0.3935
과거 질의응답:
요구사항:청년저축 계좌 해지 절차와 중도 인출 방법을 안내하시오.
고객질문:해지 승인을 받으려면 어떤 서류를 준비해서 제출해야 하는지 자세히 알려주실 수 있나요? 그리고 그 서류를 제출한 후에는 보통 처리 기간이 얼마나 소요되는지도 궁금합니다.
상담사답변:해지 승인을 위해서는 고객님의 신원을 확인할 수 있는 신분증과 최근 3개월 이내에 발급된 거래 내역서가 반드시 필요합니다. 제출하신 서류는 접수 후 영업일 기준으로 보통 1일에서 2일 정도의 시간이 소요되어 승인 절차가 완료됩니다.
꼬리질문:그러면 지금 바로 온라인으로 서류를 제출하는 것이 가능한지 알려주시겠어요? 아니면 직접 방문해서 제출해야 하는 절차가 있는지도 궁금합니다.
종합답변:해지 승인을 받기 위해서는 신분증 사본과 최근 3개월 이내 거래 내역서를 제출해야 합니다. 서류를 접수하면 영업일 기준 1~2일 안에 승인이 완료됩니다.
metadata:
- consulting_category=은행
- consulting_topic=만기,연장/해지,수신
- qa_topic=만기,연장/해지,수신
- consulting_purpose=계좌 해지 절차
---------------------------------------------------
'''